# Itinerary recommendation - run on Colab (self-contained)

Runs **Strategy A** (frozen next-POI rollout) and trains **Strategy B** (pointer network), then compares them.

**Prerequisite (from Phase 1, already on your Drive):** `/MyDrive/poi-rec/checkpoints/NYC/best.pt` and `/MyDrive/poi-rec/data/processed/NYC/`.

**How to run:** Runtime -> Change runtime type -> **T4 GPU**, then **Runtime -> Run all**. Approve the Drive popup at step 2.

## 1. Install dependencies

In [ ]:
!pip install -q torch_geometric pyarrow
print('deps installed')

## 2. Mount Drive  (approve the popup)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
PROJECT_ROOT = '/content/drive/MyDrive/poi-rec'
assert os.path.isdir(PROJECT_ROOT), 'poi-rec not found on Drive - run Phase 1 first'
print('Drive mounted; PROJECT_ROOT =', PROJECT_ROOT)

## 3. Get the code + set device/seed

In [ ]:
import sys, subprocess, importlib, random
REPO_URL = 'https://github.com/6ym6n/PFE_IMPLEMTATION.git'
REPO_DIR = '/content/PFE_IMPLEMTATION'
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git','clone','--quiet',REPO_URL,REPO_DIR], check=True)
else:
    subprocess.run(['git','-C',REPO_DIR,'pull','--quiet'], check=False)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
for _m in [k for k in list(sys.modules) if k == 'src' or k.startswith('src.')]:
    del sys.modules[_m]
importlib.invalidate_caches()
import numpy as np, torch
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('repo ready | DEVICE =', DEVICE)

## 4. Sanity check (artifacts present + GPU)

In [ ]:
import torch
print('GPU      :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE (set Runtime -> T4)')
print('best.pt  :', os.path.isfile(PROJECT_ROOT + '/checkpoints/NYC/best.pt'))
print('processed:', os.path.isdir(PROJECT_ROOT + '/data/processed/NYC'))

## 5. Strategy A - frozen next-POI rollout (no training, ~minutes)

In [ ]:
from src.itinerary.run_itinerary import run_itinerary
itin_A = run_itinerary('NYC', project_root=PROJECT_ROOT, device=DEVICE, beam=3)
gA = itin_A['greedy']['len_ge_3']['pairs-F1']
bA = itin_A['beam']['len_ge_3']['pairs-F1']
print('Strategy A len>=3 pairs-F1: greedy=%.4f | beam3=%.4f' % (gA, bA))

## 6. Strategy B - train the pointer network (~30-60 min on T4)

In [ ]:
from src.itinerary.train_pointer import train_pointer_model
model_B, test_B, history_B = train_pointer_model(
    'NYC', project_root=PROJECT_ROOT, device=DEVICE,
    epochs=50, patience=8, beam=3, min_len=3,
)

## 7. Compare A vs B  (length>=3 pairs-F1 is the headline)

In [ ]:
import json
def _load(name):
    fp = PROJECT_ROOT + '/results/' + name
    return json.load(open(fp)) if os.path.exists(fp) else None
A = _load('NYC_itinerary_greedy.json')
B = _load('NYC_pointer_test.json')
print('=== NYC itinerary - length>=3 pairs-F1 ===')
if A:
    print('  Strategy A (frozen rollout, greedy): %.4f' % A['len_ge_3']['pairs-F1'])
if B:
    bk = [k for k in B if k.startswith('beam')][0]
    print('  Strategy B (pointer, greedy)        : %.4f' % B['greedy']['pairs-F1'])
    print('  Strategy B (pointer, ' + bk + ')          : %.4f' % B[bk]['pairs-F1'])
    if A:
        print('  --> B beats the floor by: %+.4f pairs-F1' % (B['greedy']['pairs-F1'] - A['len_ge_3']['pairs-F1']))